## OPTIMAL EXECUTION

The Almgren-Chriss (2000) model of optimal execution: the closed-form
liquidation schedule that trades off market impact against timing risk.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Standard library imports

# Third-party imports
import numpy as np
import plotly.graph_objects as go
import polars as pl

# First-party imports
from xpectral.quant.execution import (
    cost_variance,
    decay_parameter,
    efficient_frontier,
    expected_cost,
    optimal_holdings,
    permanent_impact_cost,
)

## Inputs

Every input is anchored to the stock itself rather than stated as an abstract
dollar quantity: price and a percentage daily volatility combine into the
dollar volatility $\sigma$ the model needs, and a target notional combines
with price into a share count $X$. $\eta$ and $\gamma$ (impact coefficients)
and $T$ (the trading horizon) are stated directly. Three risk-aversion levels
$\lambda$ are chosen to illustrate risk-neutral, moderate, and aggressive
execution; sweeping $\lambda$ continuously later traces the full efficient
frontier.

In [3]:
price = 50.0  # $/share
sigma_pct = 0.01  # daily volatility, as a fraction of price
sigma = sigma_pct * price  # $/share/sqrt(day)

notional = 50_000.0  # $ to trade
X = notional / price  # shares

T = 1.0  # trading horizon, days
eta = 2.5e-6  # temporary impact coefficient, $*day/share^2
gamma = 2.5e-7  # permanent impact coefficient, $/share^2

# lambda: risk aversion, 1/$; kappa: decay parameter, 1/day
lambdas_illustrative = {"risk_neutral": 0.0, "moderate": 4e-5, "aggressive": 2.5e-4}
kappas_illustrative = {
    label: decay_parameter(lam, sigma, eta)
    for label, lam in lambdas_illustrative.items()
}
labels_display = {
    "risk_neutral": "Risk Neutral",
    "moderate": "Moderate",
    "aggressive": "Aggressive",
}

print(
    f"notional: ${notional:,.0f}, shares: {X:,.0f}, sigma: ${sigma:.2f}/share/sqrt(day)"
)
print("kappa by risk-aversion level:")
for label, kappa in kappas_illustrative.items():
    print(f"  {label}: {kappa:.3g}")

notional: $50,000, shares: 1,000, sigma: $0.50/share/sqrt(day)
kappa by risk-aversion level:
  risk_neutral: 0
  moderate: 2
  aggressive: 5


## Scheduled trading

The Almgren-Chriss closed-form schedule, $x(t) = X \cdot \sinh(\kappa(T-t)) /
\sinh(\kappa T)$, for the three risk-aversion levels above: risk-neutral
($\lambda=0$, a straight line), moderate, and aggressive.

In [4]:
t_grid = np.linspace(0.0, T, 101)
colors = {"risk_neutral": "#1f77b4", "moderate": "#ff7f0e", "aggressive": "#2ca02c"}

trajectories_df = pl.DataFrame(
    {
        "t_fraction": t_grid / T,
        **{
            label: optimal_holdings(t_grid, X, T, kappa)
            for label, kappa in kappas_illustrative.items()
        },
    }
)

fig = go.Figure()
for label, kappa in kappas_illustrative.items():
    legend_suffix = "λ=0" if kappa == 0.0 else f"κ={kappa:.2g}"
    fig.add_trace(
        go.Scatter(
            x=trajectories_df["t_fraction"],
            y=trajectories_df[label],
            mode="lines",
            name=f"{labels_display[label]} ({legend_suffix})",
            line={"width": 2, "color": colors[label]},
        )
    )
fig.update_layout(
    title="Optimal Holdings Trajectory x(t)",
    xaxis_title="Fraction of Time Elapsed",
    yaxis_title="Shares Held",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 0.5, "yanchor": "middle"},
)
fig.write_image("output/optimal_holdings_trajectory.png", scale=2)
fig.show()

## Cost

Expected cost splits into a permanent-impact piece, $\frac{1}{2}\gamma X^2$,
identical for every schedule since it depends only on total size $X$, and a
temporary-impact piece that shrinks as the schedule spreads trading over more
time (lower risk aversion, lower $\kappa$).

How that cost accrues over the trading horizon: cumulative permanent-impact
cost has a closed form, $(\gamma/2)(X - x(t))^2$, since it depends only on
how many shares have been sold by time $t$; cumulative temporary-impact cost
is the running integral of $\eta \dot(x)(t)^2$, computed numerically here.

In [5]:
def cumulative_trapezoid(y: np.ndarray, x: np.ndarray) -> np.ndarray:
    increments = (y[:-1] + y[1:]) / 2 * np.diff(x)
    return np.concatenate([[0.0], np.cumsum(increments)])


fig = go.Figure()
for label, kappa in kappas_illustrative.items():
    x_t = optimal_holdings(t_grid, X, T, kappa)
    trading_rate = -np.gradient(x_t, t_grid)
    cumulative_temp_cost = eta * cumulative_trapezoid(trading_rate**2, t_grid)
    cumulative_perm_cost = permanent_impact_cost(X - x_t, gamma)
    cumulative_cost = cumulative_perm_cost + cumulative_temp_cost

    legend_suffix = "λ=0" if kappa == 0.0 else f"κ={kappa:.2g}"
    fig.add_trace(
        go.Scatter(
            x=t_grid / T,
            y=cumulative_cost,
            mode="lines",
            name=f"{labels_display[label]} ({legend_suffix})",
            line={"width": 2, "color": colors[label]},
        )
    )
fig.update_layout(
    title="Cumulative Expected Cost Over the Trading Horizon",
    xaxis_title="Fraction of Time Elapsed",
    yaxis_title="Cumulative Cost ($)",
    width=700,
    height=400,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 0.02, "y": 0.98},
)
fig.show()

## Efficient frontier

Sweeping the risk-aversion parameter $\lambda$ traces out the efficient
frontier: the lowest achievable expected cost for each level of risk. Risk is
shown as the standard deviation of cost (in $, the same units as cost itself)
rather than variance ($²), which is otherwise hard to interpret at a glance.
The three risk-aversion levels above are highlighted on the curve.

In [6]:
lambdas = np.concatenate([[0.0], np.logspace(-10, -2, 49)])
frontier = efficient_frontier(lambdas, X, T, sigma, eta, gamma)
frontier_df = pl.DataFrame(
    {
        "variance": frontier["variance"],
        "expected_cost": frontier["expected_cost"],
    }
).sort("variance")
frontier_df = frontier_df.with_columns(cost_std=frontier_df["variance"].sqrt())

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=frontier_df["cost_std"],
        y=frontier_df["expected_cost"],
        mode="lines",
        name="Efficient Frontier",
        line={"width": 2},
    )
)
for label, kappa in kappas_illustrative.items():
    fig.add_trace(
        go.Scatter(
            x=[np.sqrt(cost_variance(X, sigma, T, kappa))],
            y=[expected_cost(X, gamma, eta, T, kappa)],
            mode="markers",
            name=labels_display[label],
            marker={"size": 10, "color": colors[label]},
        )
    )
fig.update_layout(
    title="Almgren-Chriss Efficient Frontier",
    xaxis_title="Std. Deviation of Cost ($)",
    yaxis_title="Cost ($)",
    width=700,
    height=450,
    margin={"l": 60, "r": 20, "t": 50, "b": 50},
    legend={"x": 1, "xanchor": "right", "y": 1, "yanchor": "top"},
)
fig.show()